[Go语言从入门到进阶实战] 第一分
========

#### 第一章 初识Golang

1.1 如下实验，原生支持并发


In [3]:
// [实验一 原生并发支持实验](https://pages.grepcode.cn/)

import ( 
    "fmt"
    "math/rand"
    "time"
) 

//数据生产者
func producer(header string, channel chan<- string) { 

    // 无限循环，不停地生产数据
    for i := 0; i < 10; i++ { 
        // 将随机数和字符串格式 为字符串发送给通道
        channel <- fmt.Sprintf ("%s: %v", header, rand.Int31()) 
    
        //等待
        time.Sleep(time.Second)
    }
    
} 

// 数据消费者
func customer(channel <- chan string) { 

    // 不停 获取数据
    for i := 0; i < 10; i++ { 
        // 从通道中取 数据，此处会阻塞直到信道中返回数据
        message := <-channel 
        
        // 打印数据
        fmt.Println(message) 
    }

} 

func main () { 
    fmt.Println("Let's rock")
    // 个字符串类型的通道
    channel := make(chan string) 
    // 创建 producer （）函数的并发 goroutine
    go producer("cat", channel) 
    go producer("dog", channel) 
    // 数据消费函数
    customer(channel) 
}

main()

Let's rock
dog: 1298498081
cat: 2019727887
dog: 1427131847
cat: 939984059
dog: 911902081
cat: 1474941318
cat: 140954425
dog: 336122540
dog: 208240456
cat: 646203300


1.2 掌握性能分析工具 Graphviz

1.3 Go从1.5版本开始自举

1.4 学习vim+dlv

#### 第二章 基本语法与使用

2.1 指针的实验

In [4]:
package main 

import ( 
    "fmt"
) 

func test01() { 
    var cat int = 1 
    
    var str string = "banana"
    
    fmt.Printf("%p %p\n", &cat, &str)
}

test01()

//交换函数
func swap(a , b *int) { 

//取a指针的值，赋给临时变量t
    t:= *a

//取b指针的值，赋给指针指向的变量
    *a = *b
 
//将a指针的值赋给指针指向的变量
    *b = t 
}

func test02() { 
 
    //准备两个变量，赋值
    x , y := 1 , 2 

    fmt.Printf("before: %v %v\n", x, y)
    //交换变量值
    swap(&x, &y) 
 
    //输出变量值
    fmt.Printf("after: %v %v\n", x, y)

}
test02()

func test03() { 
 
    str := new(string)
 
    *str = "RootMM"
    
    fmt.Printf("after: %v %v\n", str, *str)

}
test03()



0xc0000336a0 0xc0002bcfe0
before: 1 2
after: 2 1
after: 0xc0002bd0b0 RootMM


2.2 学习堆栈及内存管理

> 什么是变量逃逸
> 
> I. 逃逸分析
>
> II. 取地址发生逃逸
>
> III. 原则

In [5]:
// 2.2 变量逃逸实验
import "fmt"

//本函数测试入口参数和返回值情况
func dummy(b int) int { 
    //声明c赋值进入参数并返回
    var c int 
    c = b 
    return c 
} 

//空函数，什么也不做
func void() { 
} 

func test04() { 
 
    //声明a并打印
    var a int 

    //调用void()函数
    void() 

    //打印a的值和 dummy()函数返回
    fmt.Println(a, dummy(0))
    fmt.Println(`
$ go run -gcflags "-m -l" main.go
main.go: 25 :13 : a escapes to heap 
main.go: 25 :22 : dummy(O) escapes to heap 
main.go: 25 :13 : main ... argument does not escape
    `)
}
test04()

// 更多实验见书本，不赘述


0 0

$ go run -gcflags "-m -l" main.go
main.go: 25 :13 : a escapes to heap 
main.go: 25 :22 : dummy(O) escapes to heap 
main.go: 25 :13 : main ... argument does not escape
    


2.3 枚举的使用

In [6]:
type Weapon int

const (
    Arrow Weapon = iota // 默认值为0
    Shuriken
    SniperRifle
    Rifle
    Blower
)

func test05() {
    
    // 输出所有枚举
    fmt.Println(Arrow, Shuriken, SniperRifle, Rifle, Blower)
    
    // 使用枚举类型并赋值
    var weapon Weapon = Blower
    fmt.Println(weapon)

}

test05()

0 1 2 3 4
4


#### 第三章 掌握容器（container）的使用

删除Slice中的元素

> Go 语言中切片元素的删除过程并没有提供任何的语法糖或者方法封装，无论是初学者学习，还是实际使用都是极为麻烦的连续容器的元素删除无论是在任何语言中，都要将删除点前后的元素移动到新的位置。随着元素的增加，这个过程将会变得极为耗时。因此，当业务需要大量、频繁地从一个切片中删除元素时，如果对性能要求较高，就需要反思是否需要更换其他的容器（如双链表等能快速从删除点删除元素）

大多数语言中映射关系容器使用两种算法：散列表和平衡树。

In [32]:
import "strconv"

// container 实验

// append操作与扩容
func test06() {
    var numbers []int 
    for i := 0; i < 10 ; i++ { 
        numbers = append (numbers, i)
        fmt.Printf("len: %d cap: %d pointer: %p\n", len(numbers), cap(numbers), numbers)
    }
}

test06()

// * copy操作与map
// 遗憾，没有找到 copy map的方法，只能自己实现。

// test07
// 并发问题
import "container/list" 

func test07() { 
    l :=list.New() 

    //尾部添
    l.PushBack ("canon")
    for i := l.Front(); i != nil; i = i.Next() {
            fmt.Printf("%v ", i.Value)
    }
    fmt.Println("")
    

    //头部添加
    l.PushFront(67)
    for i := l.Front(); i != nil; i = i.Next() {
            fmt.Printf("%v ", i.Value)
    }
    fmt.Println("")
    
    //尾部添加后保存元素句柄
    element := l.PushBack ("first") 
    for i := l.Front(); i != nil; i = i.Next() {
            fmt.Printf("%v ", i.Value)
    }
    fmt.Println("")
    
    //在first之后添加 high
    l.InsertAfter("high", element)
    for i := l.Front(); i != nil; i = i.Next() {
            fmt.Printf("%v ", i.Value)
    }
    fmt.Println("")

    //在first之前添加noon
    l.InsertBefore("noon", element) 
    for i := l.Front(); i != nil; i = i.Next() {
            fmt.Printf("%v ", i.Value)
    }
    fmt.Println("")

    //使用
    l.Remove(element) 
    for i := l.Front(); i != nil; i = i.Next() {
            fmt.Printf("%v ", i.Value)
    }
    fmt.Println("")
}

test07()


len: 1 cap: 1 pointer: 0xc0000337e8
len: 2 cap: 2 pointer: 0xc000033800
len: 3 cap: 4 pointer: 0xc0000347a0
len: 4 cap: 4 pointer: 0xc0000347a0
len: 5 cap: 8 pointer: 0xc000462300
len: 6 cap: 8 pointer: 0xc000462300
len: 7 cap: 8 pointer: 0xc000462300
len: 8 cap: 8 pointer: 0xc000462300
len: 9 cap: 16 pointer: 0xc000a9e500
len: 10 cap: 16 pointer: 0xc000a9e500
canon 
67 canon 
67 canon first 
67 canon first high 
67 canon noon first high 
67 canon noon high 


#### 第四章 流程控制

`for`的功能

- Go语言的`for`包含初始化语句、条件表达式、结束语句，这3个部分均可缺省。
- `for range`支持对数组、切片、字符串、map 、channel进行遍历操作。
- 在需要时，可以使用`_`匿名变量对`for range`的变量进行选取。

`goto`的使用

> 略：在错误处理时切入错误处理的代码段
